# RL 강화학습 활용예제: OpenAI Gym 시리즈1. 막대 중심잡기
        
* 강화학습의 구현원리를 이해할 수 있는 OpenAI의 '막대 중심잡기' 예제
* 저자: RJBrooker https://github.com/RJBrooker/Q-learning-demo-Cartpole-V1
* 강연: 동준상 (naebon1@gmail.com) / 2021.1.14 / KIDET 한국국방기술학회 인공지능 세미나
* ! 이번 소스는 구글 코랩에서 실행될 때 몇 가지 문제가 발생하므로, 현재는 아나콘다 환경에서 실행만 가능

* 211112 / KPC AI 알고리즘 활용\n
* 211211 / KIDET AI

# Cartpole 예제 개요

* A pole is attached by an un-actuated joint to a cart, which moves along
* a frictionless track. The pendulum starts upright, and the goal is to
* prevent it from falling over by increasing and reducing the cart's velocity.
* This environment corresponds to the version of the cart-pole problem
* described by Barto, Sutton, and Anderson
* Reinforcement Learning: An Introduction - Stanford University
* https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf

# 에피소드 종료 조건 / Episode Termination:

* Pole Angle is more than 12 degrees.
* Cart Position is more than 2.4 (center of the cart reaches the edge of the display).
* Episode length is greater than 200.
* Solved Requirements:
  - Considered solved when the average return is greater than or equal to 195.0 over 100 consecutive trials.

# 라이브러리 설치 및 임포트

In [1]:
#!pip install --upgrade pip
!pip install gymnasium

import gymnasium as gym

import numpy as np 
import time, math, random
from typing import Tuple

  Using cached gymnasium-1.2.3-py3-none-any.whl.metadata (10 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached Farama_Notifications-0.0.4-py3-none-any.whl.metadata (558 bytes)
Using cached gymnasium-1.2.3-py3-none-any.whl (952 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached Farama_Notifications-0.0.4-py3-none-any.whl (2.5 kB)


In [2]:
!pip install scikit-learn

# KBinsDiscretizer를 임포트하지 못하는 경우, 콘솔에서 conda update scikit-learn 실행
from sklearn.preprocessing import KBinsDiscretizer

# CartPole-v1

In [3]:
env = gym.make('CartPole-v1')

In [4]:
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>

# 학습 전 에이전트의 동작 확인 및 실행환경 시각화 / Visualise Enviroment

* Visualise the eniroment/simulation

In [7]:
policy = lambda obs: 1
range_num_step1 = 1#5
range_num_step2 = 3#80
done_true = False
last_obs = []
last_reward = 0.0
last_info = False
last_actions = None
last_i1 = last_i0 = 0
for i0 in range(range_num_step1):
    obs = env.reset()
    for i1 in range(range_num_step2):
        # actions = policy(obs)
        actions = env.action_space.sample()
        obs, reward, done, info, _ = env.step(actions)
        print(f'i={i1}, obs={obs}, reward={reward}, done={done}, info={info}, actions={actions}')
        if done == True:
            last_i1 = i1
            last_actions = actions
            last_info = info
            last_reward = reward
            done_true = True
            break
        env.render()
        time.sleep(0.05)
    if done_true == True:
        last_i0 = i0
        break
print(f'i0={last_i0}, i1={last_i1}, obs={last_obs}, reward={last_reward}, info={last_info}, actions={last_actions}')
env.close()

i=0, obs=[-0.03225517  0.16828719  0.00452608 -0.24948505], reward=1.0, done=False, info=False, actions=1
i=1, obs=[-0.02888943 -0.02689911 -0.00046362  0.04462205], reward=1.0, done=False, info=False, actions=0
i=2, obs=[-0.02942741  0.16822949  0.00042882 -0.24820712], reward=1.0, done=False, info=False, actions=1
i0=0, i1=0, obs=[], reward=0.0, info=False, actions=None


/home/simpson/work/mlops_jupyter_nodebooks/jupyter_venv/lib/python3.12/site-packages/gymnasium/envs/classic_control/cartpole.py:250: UserWarning: WARN: You are calling render method without specifying any render mode. You can specify the render_mode at initialization, e.g. gym.make("CartPole-v1", render_mode="rgb_array")
  gym.logger.warn(
